# Flowers-102 MaskMix + VPT — experiments notebook

Runs on Colab T4 via the Cursor remote-kernel bridge. Cell 1 clones the repo and mounts Drive; subsequent cells drive `src.train.train_one_config` from YAML configs. Later cells (Tasks 12–15.5) will be appended as experiment blocks are rolled out.

In [ ]:
# %% Cell 1 — environment bootstrap (paths, sys.path, sanity print)
import os, sys
from pathlib import Path

# Three env vars drive all paths. Defaults below target vast.ai / H100 SSH layout.
# Local laptop: leaves all three unset; the CWD fallback for SC4001_REPO kicks in.
# Colab: set SC4001_DATA / SC4001_CKPT explicitly if you want Drive persistence.
repo_root_default = "/workspace/SC4001-FinalProject"
if not Path(repo_root_default).exists():
    cwd = Path().resolve()
    repo_root_default = str(cwd.parent if cwd.name == "notebooks" else cwd)

REPO_ROOT = Path(os.environ.get("SC4001_REPO", repo_root_default)).resolve()
DATA_ROOT = Path(os.environ.get("SC4001_DATA", "/workspace/data/flowers-102"))
CKPT_ROOT = Path(os.environ.get("SC4001_CKPT", "/workspace/checkpoints"))

sys.path.insert(0, str(REPO_ROOT))

RESULTS_JSONL = REPO_ROOT / "results" / "runs.jsonl"
FIGURES_DIR = REPO_ROOT / "figures"
for p in (CKPT_ROOT, DATA_ROOT, RESULTS_JSONL.parent, FIGURES_DIR):
    p.mkdir(parents=True, exist_ok=True)

import torch
print(f"Repo : {REPO_ROOT}")
print(f"Data : {DATA_ROOT}")
print(f"Ckpts: {CKPT_ROOT}")
print(f"Torch: {torch.__version__}  CUDA: {torch.cuda.is_available()}  "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")


In [ ]:
# %%
from scripts.download_masks import download
download(DATA_ROOT)

In [ ]:
# %%
import numpy as np
import matplotlib.pyplot as plt
from src.data import Flowers102WithMasks

ds = Flowers102WithMasks(root=DATA_ROOT, split="train", image_size=224, train_augment=False)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, idx in zip(axes, rng.integers(0, len(ds), 5)):
    img, mask, label = ds[int(idx)]
    # Unnormalise
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_show = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_show)
    ax.imshow(mask[0].numpy(), alpha=0.4, cmap="Reds")
    ax.set_title(f"class {label}")
    ax.axis("off")
plt.suptitle("Mask / image alignment sanity check")
plt.savefig(FIGURES_DIR / "sanity_mask_alignment.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# %%
from src.train import train_one_config

result_D = train_one_config(
    config_path=REPO_ROOT / "configs/D_linear_probe.yaml",
    seed=0,
    data_root=DATA_ROOT,
    checkpoint_dir=CKPT_ROOT,
    results_path=RESULTS_JSONL,
)
print(result_D)

In [ ]:
# %%
from src.train import train_one_config

BLOCK_A_CONFIGS = [
    "configs/A1_baseline.yaml",
    "configs/A2_maskmix.yaml",
    "configs/A3_attsup.yaml",
    "configs/A4_ours.yaml",
]
SEEDS = [0, 1, 2]

block_a_results = []
for cfg in BLOCK_A_CONFIGS:
    for seed in SEEDS:
        print(f"\n=== {cfg} seed={seed} ===")
        r = train_one_config(
            config_path=REPO_ROOT / cfg,
            seed=seed,
            data_root=DATA_ROOT,
            checkpoint_dir=CKPT_ROOT,
            results_path=RESULTS_JSONL,
        )
        block_a_results.append(r)
        print(r)

import pandas as pd
df_a = pd.DataFrame(block_a_results)
df_a.to_csv(REPO_ROOT / "results/block_a.csv", index=False)
df_a

In [ ]:
# %%
block_c_results = []
for seed in [0, 1, 2]:
    print(f"\n=== A5_cutmix_attsup seed={seed} ===")
    r = train_one_config(
        config_path=REPO_ROOT / "configs/A5_cutmix_attsup.yaml",
        seed=seed,
        data_root=DATA_ROOT,
        checkpoint_dir=CKPT_ROOT,
        results_path=RESULTS_JSONL,
    )
    block_c_results.append(r)
    print(r)

df_c = pd.DataFrame(block_c_results)
df_c.to_csv(REPO_ROOT / "results/block_c.csv", index=False)
df_c

In [ ]:
# %%
block_b_configs = [
    ("k1", "configs/B_k1_baseline.yaml"),
    ("k1", "configs/B_k1_ours.yaml"),
    ("k5", "configs/B_k5_baseline.yaml"),
    ("k5", "configs/B_k5_ours.yaml"),
]
block_b_results = []
for tag, cfg in block_b_configs:
    print(f"\n=== {cfg} seed=0 ===")
    r = train_one_config(
        config_path=REPO_ROOT / cfg,
        seed=0,
        data_root=DATA_ROOT,
        checkpoint_dir=CKPT_ROOT,
        results_path=RESULTS_JSONL,
    )
    r["k_tag"] = tag
    block_b_results.append(r)
    print(r)

df_b = pd.DataFrame(block_b_results)
df_b.to_csv(REPO_ROOT / "results/block_b.csv", index=False)
df_b

In [ ]:
# %%
import pandas as pd

df_a = pd.read_csv(REPO_ROOT / "results/block_a.csv")
config_name_map = {
    "A1_baseline_seed0": "A1 baseline", "A1_baseline_seed1": "A1 baseline", "A1_baseline_seed2": "A1 baseline",
    "A2_maskmix_seed0":  "A2 +MaskMix", "A2_maskmix_seed1":  "A2 +MaskMix", "A2_maskmix_seed2":  "A2 +MaskMix",
    "A3_attsup_seed0":   "A3 +AttSup",  "A3_attsup_seed1":   "A3 +AttSup",  "A3_attsup_seed2":   "A3 +AttSup",
    "A4_ours_seed0":     "A4 ours",     "A4_ours_seed1":     "A4 ours",     "A4_ours_seed2":     "A4 ours",
}
df_a["config"] = df_a["run"].map(config_name_map)
headline = df_a.groupby("config").agg(
    top1_mean=("test_top1", "mean"),
    top1_std=("test_top1", "std"),
    per_class_mean=("test_per_class_mean", "mean"),
    per_class_std=("test_per_class_mean", "std"),
).round(4)
headline.to_csv(REPO_ROOT / "results/headline_table.csv")
headline

In [ ]:
# %%
import numpy as np
from torch.utils.data import DataLoader
from src.bootstrap import paired_bootstrap_pvalue
from src.data import Flowers102WithMasks
from src.model import VPTDeepViT

def per_example_correct(ckpt_path, data_root):
    state = torch.load(ckpt_path, map_location="cuda" if torch.cuda.is_available() else "cpu")
    cfg = state["cfg"]
    model = VPTDeepViT(num_prompts=cfg.get("num_prompts", 10), num_classes=102,
                       capture_last_layers=cfg.get("capture_last_layers", 2))
    model.load_state_dict(state["model"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    test_ds = Flowers102WithMasks(root=data_root, split="test", image_size=cfg["image_size"],
                                  train_augment=False)
    loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
    preds, labels = [], []
    with torch.no_grad():
        for x, _m, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                logits, _ = model(x, return_attn=False)
            preds.append(logits.argmax(1).cpu())
            labels.append(y.cpu())
    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    return (preds == labels)

a1 = per_example_correct(CKPT_ROOT / "A1_baseline_seed0" / "best.pt", DATA_ROOT)
a4 = per_example_correct(CKPT_ROOT / "A4_ours_seed0" / "best.pt", DATA_ROOT)
mean_diff, p = paired_bootstrap_pvalue(a4, a1, n_resamples=5000, seed=0)
print(f"A4 - A1 mean diff = {mean_diff:+.4f}, two-sided p = {p:.4f}")

with open(REPO_ROOT / "results/significance_A4_vs_A1.txt", "w") as f:
    f.write(f"A4 - A1 mean diff = {mean_diff:+.4f}\n")
    f.write(f"two-sided paired bootstrap p = {p:.4f}\n")
    f.write(f"n_resamples = 5000, seed = 0\n")

In [ ]:
# %%
import matplotlib.pyplot as plt

df_b = pd.read_csv(REPO_ROOT / "results/block_b.csv")
# Block A seed-0 for k=10
df_a = pd.read_csv(REPO_ROOT / "results/block_a.csv")
a1_k10 = df_a[df_a["run"].str.startswith("A1_baseline")]
a4_k10 = df_a[df_a["run"].str.startswith("A4_ours")]

ks = [1, 5, 10]
baseline_means = [
    df_b[df_b["run"] == "B_k1_baseline_seed0"]["test_top1"].iloc[0],
    df_b[df_b["run"] == "B_k5_baseline_seed0"]["test_top1"].iloc[0],
    a1_k10["test_top1"].mean(),
]
baseline_stds = [0.0, 0.0, a1_k10["test_top1"].std()]
ours_means = [
    df_b[df_b["run"] == "B_k1_ours_seed0"]["test_top1"].iloc[0],
    df_b[df_b["run"] == "B_k5_ours_seed0"]["test_top1"].iloc[0],
    a4_k10["test_top1"].mean(),
]
ours_stds = [0.0, 0.0, a4_k10["test_top1"].std()]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.errorbar(ks, baseline_means, yerr=baseline_stds, marker="o", label="A1 baseline")
ax.errorbar(ks, ours_means, yerr=ours_stds, marker="s", label="A4 ours")
ax.set_xlabel("training images per class (k)")
ax.set_ylabel("test top-1 accuracy")
ax.set_xticks(ks)
ax.set_title("Data-efficiency curve")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "k_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %%
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

from src.data import Flowers102WithMasks
from src.model import VPTDeepViT

def load_model(ckpt_path, device):
    state = torch.load(ckpt_path, map_location=device)
    cfg = state["cfg"]
    m = VPTDeepViT(num_prompts=cfg.get("num_prompts", 10), num_classes=102,
                   capture_last_layers=cfg.get("capture_last_layers", 2))
    m.load_state_dict(state["model"])
    return m.to(device).eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
m_a1 = load_model(CKPT_ROOT / "A1_baseline_seed0" / "best.pt", device)
m_a4 = load_model(CKPT_ROOT / "A4_ours_seed0"     / "best.pt", device)

# Pick three test-set samples from classes known to be confused in Nilsback Fig. 6.
# Class names follow the Flowers-102 "name.py" list (0-indexed).
# Tree Mallow (82), Pink Primrose (0), Petunia (53), Garden Phlox (19),
# Water Lily (70), Siam Tulip (99).
TARGETS = [0, 53, 70]  # pick one from each confusion pair

test_ds = Flowers102WithMasks(root=DATA_ROOT, split="test", image_size=224, train_augment=False)

def first_index_of_class(ds, c):
    for i in range(len(ds)):
        if ds[i][2] == c:
            return i
    raise ValueError(f"class {c} not found")

fig, axes = plt.subplots(len(TARGETS), 4, figsize=(12, 3 * len(TARGETS)))
for row, c in enumerate(TARGETS):
    idx = first_index_of_class(test_ds, c)
    img, mask, label = test_ds[idx]
    x = img.unsqueeze(0).to(device)
    with torch.no_grad():
        _, attn_a1 = m_a1(x, return_attn=True)
        _, attn_a4 = m_a4(x, return_attn=True)
    def to_grid(a):
        return a.view(14, 14).cpu().numpy()

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_show = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[row, 0].imshow(img_show); axes[row, 0].set_title(f"class {label}")
    axes[row, 1].imshow(mask[0].numpy(), cmap="gray"); axes[row, 1].set_title("GT mask")
    axes[row, 2].imshow(img_show); axes[row, 2].imshow(to_grid(attn_a1[0]), alpha=0.6, cmap="hot")
    axes[row, 2].set_title("A1 baseline attn")
    axes[row, 3].imshow(img_show); axes[row, 3].imshow(to_grid(attn_a4[0]), alpha=0.6, cmap="hot")
    axes[row, 3].set_title("A4 ours attn")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "qualitative_attention.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# %% Cell 12 — push results back to GitHub (opt-in via SC4001_PUSH)
import os, subprocess

if not os.environ.get("SC4001_PUSH"):
    print("SC4001_PUSH not set; skipping git push. Export SC4001_PUSH=1 and SC4001_GH_TOKEN to enable.")
else:
    token = os.environ.get("SC4001_GH_TOKEN")
    if not token:
        raise RuntimeError(
            "SC4001_PUSH is set but SC4001_GH_TOKEN is missing. "
            "Export both SC4001_PUSH=1 and SC4001_GH_TOKEN=<your PAT> to push."
        )

    def run(cmd, cwd=None):
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
        print(p.stdout, p.stderr)
        return p.returncode

    # Set git identity (must happen once per session).
    run(["git", "-C", str(REPO_ROOT), "config", "user.email", "truongdanggiahuy2004@gmail.com"])
    run(["git", "-C", str(REPO_ROOT), "config", "user.name",  "huytruong2004"])

    # Stage results + figures (not checkpoints, too large).
    run(["git", "-C", str(REPO_ROOT), "add", "results/", "figures/", "notebooks/"])
    run(["git", "-C", str(REPO_ROOT), "commit", "-m", "results: add experiment outputs from remote session"])

    # Push with PAT from env.
    push_url = f"https://{token}@github.com/huytruong2004/SC4001-FinalProject.git"
    run(["git", "-C", str(REPO_ROOT), "push", push_url, "main"])
    print("Done. Re-check the repo on GitHub to confirm.")
